# Execução das próximas etapas — APAC e SIH/SUS

Este notebook registra o encerramento da auditoria interna APAC/SIH e as pendências que exigem fonte oficial, validação técnica ou decisão clínica. Os DBCs brutos são preservados; A fonte de microdados restrita a N18 permanece fora dos totais finais; N18 continua incluído na categoria TabNet N17–N19.

**Situação atual (15/11/2026):** a reextração N18 é coerente como subconjunto de N17–N19 nas células TabNet preenchidas; a diferença histórica de 12 linhas não é reproduzível e não bloqueia o painel. Datas, continuidade e identificador APAC seguem sem validação para coorte; comparador clínico ainda depende de decisão especializada.

> **Atualização da entrega final:** a extração SIH/SIA revisada já foi incorporada ao [notebook 09](09_ANALISE_ESTATISTICA_E_PREVISAO_DRC.ipynb), com RR/dezembro=14 observado e tendência linear SIH avaliada. A confirmação administrativa de encerramento continua externa. Este caderno preserva os resultados da auditoria APAC/N18 e não integra uma coorte; os 647 pares citados pertencem ao recorte histórico de reconciliação.


In [1]:
from pathlib import Path
import csv

candidates = [Path.cwd() / "Trabalho-final", Path.cwd(), Path.cwd().parent]
project_dir = next(p.resolve() for p in candidates if (p / "Dados").is_dir())
data_dir = project_dir / "Dados"
audit_dir = data_dir / "02_derivados_preliminares/APAC/auditoria_excecoes_20261105_v2"
docs_dir = data_dir / "04_documentacao_e_auditoria"

def read_rows(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        sample = f.read(4096)
        f.seek(0)
        dialect = csv.Sniffer().sniff(sample, delimiters=",;")
        return list(csv.DictReader(f, dialect=dialect))

def show(rows, limit=30):
    for row in rows[:limit]:
        print(" | ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} linhas adicionais")

print(f"Projeto: {project_dir}")
print(f"Auditoria APAC: {audit_dir}")

Projeto: /Users/paulis/Documents/Projects/POS-SENAI/estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final
Auditoria APAC: /Users/paulis/Documents/Projects/POS-SENAI/estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados/02_derivados_preliminares/APAC/auditoria_excecoes_20261105_v2


## Etapa 1 — Reproduzir os achados APAC já auditados

A célula seguinte confere artefatos agregados da auditoria e exibe somente contagens ou valores de data. Não relê os DBCs nem mostra identificadores de paciente ou autorização.

In [2]:
resumo = read_rows(audit_dir / "resumo.csv")
hashes = read_rows(audit_dir / "hashes_apac.csv")
por_ano = read_rows(audit_dir / "datas_futuras_por_ano.csv")
por_comp = read_rows(audit_dir / "datas_futuras_por_competencia_apac.csv")
valores_data = read_rows(audit_dir / "valores_datas_futuras.csv")
resumo_dict = {r["indicador"]: int(r["valor"]) for r in resumo}
assert len(hashes) == 648
assert resumo_dict["registros"] == 3_186_282
assert resumo_dict["tp4_ou_procedimento_vazio"] == 12
assert resumo_dict["ATD_DTPDR_apos_2025"] == 146
assert resumo_dict["ATD_DTCLI_apos_2025"] == 1_162
print("Artefatos agregados conferidos: 648 arquivos; 3.186.282 registros; exceções reproduzidas.")
show(resumo)
print("Distribuição de datas por ano:")
show(por_ano, limit=100)

Artefatos agregados conferidos: 648 arquivos; 3.186.282 registros; exceções reproduzidas.
indicador: ATD_DTCLI_apos_2025 | valor: 1162
indicador: ATD_DTPDR_apos_2025 | valor: 146
indicador: registros | valor: 3186282
indicador: tp4_ou_procedimento_vazio | valor: 12
Distribuição de datas por ano:
campo: ATD_DTCLI | ano: 2026 | registros: 247
campo: ATD_DTCLI | ano: 2027 | registros: 338
campo: ATD_DTCLI | ano: 2028 | registros: 138
campo: ATD_DTCLI | ano: 2029 | registros: 108
campo: ATD_DTCLI | ano: 2031 | registros: 4
campo: ATD_DTCLI | ano: 2032 | registros: 21
campo: ATD_DTCLI | ano: 2033 | registros: 3
campo: ATD_DTCLI | ano: 2034 | registros: 3
campo: ATD_DTCLI | ano: 2042 | registros: 3
campo: ATD_DTCLI | ano: 2043 | registros: 2
campo: ATD_DTCLI | ano: 2045 | registros: 2
campo: ATD_DTCLI | ano: 2046 | registros: 1
campo: ATD_DTCLI | ano: 2047 | registros: 4
campo: ATD_DTCLI | ano: 2051 | registros: 4
campo: ATD_DTCLI | ano: 2052 | registros: 17
campo: ATD_DTCLI | ano: 2055 | re

In [3]:
# Datas sem chave pessoal. A auditoria de origem já verificou formato/calendário;
# esta célula apenas resume valores e não valida significado clínico.
valores_data.sort(key=lambda r: (r["campo"], -int(r["registros"])))
show(valores_data, limit=25)

campo: ATD_DTCLI | valor_data: 20260228 | registros: 34
campo: ATD_DTCLI | valor_data: 20270327 | registros: 30
campo: ATD_DTCLI | valor_data: 20290306 | registros: 24
campo: ATD_DTCLI | valor_data: 22021011 | registros: 24
campo: ATD_DTCLI | valor_data: 20270227 | registros: 23
campo: ATD_DTCLI | valor_data: 20270720 | registros: 21
campo: ATD_DTCLI | valor_data: 20260131 | registros: 20
campo: ATD_DTCLI | valor_data: 20260611 | registros: 19
campo: ATD_DTCLI | valor_data: 40240404 | registros: 18
campo: ATD_DTCLI | valor_data: 20270716 | registros: 17
campo: ATD_DTCLI | valor_data: 20281228 | registros: 17
campo: ATD_DTCLI | valor_data: 20290913 | registros: 17
campo: ATD_DTCLI | valor_data: 29240829 | registros: 17
campo: ATD_DTCLI | valor_data: 20270601 | registros: 12
campo: ATD_DTCLI | valor_data: 20520603 | registros: 12
campo: ATD_DTCLI | valor_data: 20270315 | registros: 11
campo: ATD_DTCLI | valor_data: 20260722 | registros: 10
campo: ATD_DTCLI | valor_data: 20261101 | regist

**Leitura:** as datas anômalas são raras, mas incluem anos muito distantes; baixa frequência não elimina viés em uma coorte. Manter `ATD_DTPDR` e `ATD_DTCLI` fora de tempo zero e seguimento. Os DBCs não foram corrigidos nem filtrados.

## Etapa 2 — Conferir documentação ATDAC

O Informe Técnico SIASUS 2019.07 descreve os campos do arquivo dialítico: `ATD_DTPDR` = início da primeira diálise; `ATD_DTCLI` = início da diálise na clínica; ambos como `AAAAMMDD`. A listagem oficial atual disponibiliza layout genérico APAC, sem esclarecer a codificação dos anos futuros observados. A definição nominal está documentada; a validade semântica das datas anômalas continua pendente. Não corrigir por inferência nem usar como tempo zero/seguimento até obter esclarecimento técnico ou validação independente.

Referências: [Informe Técnico SIASUS 2019.07 (cópia consultada)](https://rfsaldanha.github.io/sis/assets/sia/Informe_Tecnico_SIASUS_2019_07.pdf); [documentos APAC do DATASUS](https://sia.datasus.gov.br/documentos/listar_ftp_apac.php).

In [4]:
# Distribuição das exceções por competência APAC, com zeros preenchidos.
campos = sorted({r["campo"] for r in por_comp})
competencias = sorted({r["AP_CMP"] for r in por_comp})
contagens = {(r["AP_CMP"], r["campo"]): int(r["registros"]) for r in por_comp}
print("AP_CMP | " + " | ".join(campos))
for comp in competencias[-12:]:
    print(comp + " | " + " | ".join(str(contagens.get((comp, campo), 0)) for campo in campos))

AP_CMP | ATD_DTCLI | ATD_DTPDR
202501 | 45 | 6
202502 | 47 | 6
202503 | 51 | 3
202504 | 48 | 4
202505 | 50 | 4
202506 | 50 | 6
202507 | 55 | 11
202508 | 57 | 10
202509 | 53 | 10
202510 | 51 | 9
202511 | 71 | 15
202512 | 99 | 20


## Etapa 3 — Continuidade e identificador longitudinal

A auditoria agregada de 05/11/2026 encontrou 1.199.885 números únicos de autorização no recorte e 12.065 autorizações associadas a mais de um identificador pseudonimizado. Esse achado não estabelece se a causa é continuidade, transferência, troca de identificador ou inconsistência. `AP_AUTORIZ` não deve ser usado como chave de pessoa.

A análise de `AP_APACANT` encontrou 81.416 pares distintos autorização atual→anterior: 40.170 autorreferências e 10.756 sem alvo no recorte. Entre 77.203 pares com datas válidas, 41.587 iniciam antes do fim registrado da referência e 35.608 depois. Além disso, 1.953.456 linhas `AP_TPAPAC=2` não têm referência anterior preenchida. Esses achados não confirmam continuidade clínica.

**Estado: pendente de regra técnica e validação de estabilidade do identificador.** Não montar coorte longitudinal até definir início, continuidade, transferências, trocas de modalidade e ausência/colisão do identificador.


### Rechecagem agregada das referências APAC

A rotina auditou `AP_APACANT` sem exportar números de autorização ou CNS. Esta célula confere apenas os agregados e explicita por que eles não autorizam uma cadeia de pacientes.

In [5]:
continuity_dir = data_dir / "02_derivados_preliminares/APAC/auditoria_continuidade_20261105_v1"
chain_summary = {r["indicador"]: int(r["valor"]) for r in read_rows(continuity_dir / "resumo.csv")}
chain_refs = {r["indicador"]: int(r["valor"]) for r in read_rows(continuity_dir / "referencias_apac_agregadas.csv")}
linhas_por_tipo = {r["AP_TPAPAC"]: int(r["linhas"]) for r in read_rows(continuity_dir / "distribuicao_linhas_por_tipo_apac.csv")}
assert chain_summary["linhas_lidas"] == 3_186_282
assert chain_summary["autorizacoes_unicas"] == 1_199_885
assert chain_summary["linhas_com_apac_anterior"] == 149_406
assert chain_refs["referencias_unicas"] == 81_416
assert chain_refs["referencia_propria"] == 40_170
assert chain_refs["alvo_nao_encontrado_no_recorte"] == 10_756
assert chain_summary["autorizacoes_com_multiplos_ids_no_conjunto"] == 12_065
print("Linhas lidas:", chain_summary["linhas_lidas"])
print("Autorizações únicas:", chain_summary["autorizacoes_unicas"])
print("Linhas com APAC anterior:", chain_summary["linhas_com_apac_anterior"])
print("Pares únicos atual→anterior:", chain_refs["referencias_unicas"])
print("Autorreferências:", chain_refs["referencia_propria"])
print("Referências sem alvo no recorte:", chain_refs["alvo_nao_encontrado_no_recorte"])
print("APAC TP=2 sem referência anterior preenchida:", chain_summary["linhas_tp2_sem_referencia_anterior"])
print("Autorizações associadas a múltiplos identificadores pseudonimizados:", chain_summary["autorizacoes_com_multiplos_ids_no_conjunto"])
print(f"Autorreferências entre pares distintos: {chain_refs['referencia_propria'] / chain_refs['referencias_unicas']:.1%}")
print(f"Referências sem alvo no recorte: {chain_refs['alvo_nao_encontrado_no_recorte'] / chain_refs['referencias_unicas']:.1%}")
print(f"APAC tipo 2 sem referência anterior: {chain_summary['linhas_tp2_sem_referencia_anterior'] / linhas_por_tipo['2']:.1%}")
print("Decisão: cadeia administrativa não comprova seguimento individual; nenhum identificador exportado.")

Linhas lidas: 3186282
Autorizações únicas: 1199885
Linhas com APAC anterior: 149406
Pares únicos atual→anterior: 81416
Autorreferências: 40170
Referências sem alvo no recorte: 10756
APAC TP=2 sem referência anterior preenchida: 1953456
Autorizações associadas a múltiplos identificadores pseudonimizados: 12065
Autorreferências entre pares distintos: 49.3%
Referências sem alvo no recorte: 13.2%
APAC tipo 2 sem referência anterior: 94.9%
Decisão: cadeia administrativa não comprova seguimento individual; nenhum identificador exportado.


## Etapa 4 — SIH: decisão de uso

A reextração dos DBCs encontrou 213.789 linhas N18: 213.788 com `IDENT=1` e uma com `IDENT=5`. Nas 647 células TabNet observadas, N18 ficou abaixo de N17–N19, coerente com o recorte diagnóstico mais amplo. O CSV antigo tem 368.917 linhas, 155.128 a mais que a reextração; essa é a divergência principal. A diferença histórica de 12 linhas ocorreu entre duas rotinas antigas e não explica o contraste atual.

**Decisão:** os microdados restritos a N18 permanecem preliminares e fora dos totais finais. Não se busca igualdade entre N18 e N17–N19, não se deduplica por `N_AIH` e não se contam registros como pacientes. A reextração e os cenários novos estão em `Dados/02_derivados_preliminares/SIH/auditoria_sih_dbc_reconciliacao_20261105_221707/`; detalhes e definição TabNet na [nota SIH](../../docs/Trabalho-final/Dados/04_documentacao_e_auditoria/LEIA-ME_SIH_N18_2024_2025.md).


## Etapa 5 — Pergunta comparativa para validação clínica

Rascunho para discussão: hemodiálise versus diálise peritoneal entre pacientes clinicamente elegíveis para ambas, com população, início de exposição, desfecho e horizonte de seguimento a definir com nefrologista e epidemiologista clínico.

A alternativa diálise versus cuidado conservador exige uma fonte de pacientes elegíveis não dialíticos; ela não foi identificada nas bases locais. A APAC atual não comprova, sozinha, seguimento de mortalidade completo.

**Estado: rascunho, não aprovado e não estimável ainda.** Abrir [protocolo provisório](../../docs/Trabalho-final/protocolo_viabilidade_comparativa_provisorio.md) e registrar decisão clínica antes de construir coorte.

In [6]:
# Quadro de andamento; atualizar status após cada decisão documentada.
status = [
    {"etapa": "Exceções/datas APAC", "status": "Datas sintaticamente válidas, mas semanticamente pendentes"},
    {"etapa": "Layout técnico ATDAC", "status": "Layout 2019 encontrado; anomalias dependem de esclarecimento DATASUS"},
    {"etapa": "Identificador e continuidade", "status": "Pendente de validação"},
    {"etapa": "SIH — recortes divergentes", "status": "CSV antigo excede reextração em 155.128 linhas; N18 preliminar"},
    {"etapa": "Comparador e estimando", "status": "Rascunho; requer avaliação clínica"},
    {"etapa": "Coorte e estimativa de efeito", "status": "Não iniciar até cumprir critérios de viabilidade"},
]
show(status)

etapa: Exceções/datas APAC | status: Datas sintaticamente válidas, mas semanticamente pendentes
etapa: Layout técnico ATDAC | status: Layout 2019 encontrado; anomalias dependem de esclarecimento DATASUS
etapa: Identificador e continuidade | status: Pendente de validação
etapa: SIH — recortes divergentes | status: CSV antigo excede reextração em 155.128 linhas; N18 preliminar
etapa: Comparador e estimando | status: Rascunho; requer avaliação clínica
etapa: Coorte e estimativa de efeito | status: Não iniciar até cumprir critérios de viabilidade


## Próximas ações após este notebook

1. **SIH:** manter a fonte de microdados N18 fora dos totais e usar a categoria N17–N19 do TabNet no painel. A recuperação da rotina histórica serviria apenas para rastreabilidade das 12 linhas; não é pré-requisito para a análise territorial. A coleta revisada já esclareceu o corte de processamento e foi integrada no notebook 09; eventual confirmação de encerramento administrativo depende do DATASUS.
2. **APAC:** obter esclarecimento técnico sobre datas anômalas, autorreferências e referências anteriores; não recodificar por inferência.
3. **Longitudinalidade:** validar identificador, continuidade, transferências e completude de desfechos antes de qualquer coorte.
4. **Pergunta clínica:** definir comparador, elegibilidade, tempo zero, desfecho e horizonte com especialistas; avaliar fonte para elegíveis não dialíticos. O quadro de encerramento, a minuta ao DATASUS e o checklist clínico estão em `Dados/04_documentacao_e_auditoria/PENDENCIAS_EXTERNAS_E_METODOLOGICAS_2026-11-15.md`.

**Não concluir eficácia ou efetividade a partir da comparação agregada SIA/SIH.**
